In [1]:
import warnings
warnings.simplefilter(action='ignore')

import argparse
import os,cv2
import random,numpy,pandas
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [2]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [3]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

Random Seed:  999


In [4]:
ngpu=1
ngf,nc = 3,3
ndf = 64

fsc_test={}

transform = transforms.Compose([
    transforms.Resize((450,450)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

for i in os.listdir('/kaggle/input/fake-scene-dataset/test'):
    fsc_test[i]=transform(Image.open(f'/kaggle/input/fake-scene-dataset/test/{i}'))

print(fsc_test["21.jpg"].shape)
device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

torch.Size([3, 450, 450])


In [5]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 2),
            nn.Sigmoid()
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x

In [6]:
EFF_NET = EffnetModel().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)
#EFF_NET
#EFF_NET.load_state_dict(torch.load("/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/0.00015554654237348586 87.0.pth",weights_only=False,map_location=torch.device('cpu')))

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:00<00:00, 222MB/s]


In [7]:
fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")

In [8]:
sub = [0]*180
file_ = os.listdir('/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/')

for j in file_:
    print(j)
    z_add,z_total=0,0
    EFF_NET.load_state_dict(torch.load(f"/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/{j}", 
                                       weights_only=False, map_location=torch.device('cpu')))
    for i,j in zip(fsc_submission.index,range(180)):
        
        img=fsc_test[i].reshape((1, 3, 450, 450)).float().to(device)
        sub[j] += EFF_NET(img).cpu().detach().numpy()[0][1]

8.832199819153175e-05 43.pth
0.00012810539919883013 46.pth
0.00013342038437258452 26.pth
9.497477003606036e-05 48.pth
0.000153549131937325 31.pth
9.444329043617472e-05 24.pth
6.075040073483251e-05 34.pth
0.00016210344620049 25.pth
2.846710958692711e-05 49.pth
0.00011090168118244037 36.pth
5.449479431263171e-05 45.pth
2.617700374685228e-05 37.pth
0.00010551880404818803 41.pth
0.0001668319891905412 47.pth
0.00017586146714165807 30.pth
0.00019221795082557946 35.pth


In [9]:
submission=pandas.DataFrame({'image' : fsc_submission.index, 
                             'label' : numpy.array(sub)/len(file_)})
pandas.DataFrame(submission).to_csv(f"submission.csv", index=False)
pandas.DataFrame(submission)

,image,label
0,102.jpg,0.041396
1,108.jpg,0.996004
2,109.jpg,0.997872
3,111.jpg,0.022130
4,121.jpg,0.007647
...,...,...
175,899.jpg,0.033743
176,91.jpg,0.987483
177,94.jpg,0.998774
178,95.jpg,0.999026
